<a href="https://colab.research.google.com/github/matti410/Trading-System-Creator_V4/blob/main/Trading_System_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trading System Lab
### Dall'idea alla strategia verificata, in sette passi

Scegli tu i segnali di ingresso e le uscite. Lo strumento misura, scarta quello che non regge e ti dice quanto rischio aspettarti.

**Come si usa:** in ogni passo imposta i valori nel riquadro a destra, poi premi il tasto ▶ a sinistra della cella. I passi vanno eseguiti in ordine, dall'alto verso il basso.

## Avvio
Da eseguire una volta sola, all'inizio. Richiede circa un minuto.

In [ ]:
#@title Avvio { display-mode: "form" }
import os, shutil, subprocess, sys, warnings
warnings.simplefilter("ignore")

REPO = "matti410/Trading-System-Creator_V4"
CARTELLA = "/content/Trading-System-Creator_V4"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "TA-Lib==0.8.1", "backtesting==0.6.6"], check=True)

# Il token di GitHub si legge dai Secrets di Colab (icona della chiave, a sinistra):
# non va mai scritto qui dentro. Senza token si prova il repo pubblico.
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

os.chdir("/content")
shutil.rmtree(CARTELLA, ignore_errors=True)
indirizzo = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
esito = subprocess.run(["git", "clone", "--quiet", "--depth", "1", indirizzo, CARTELLA], capture_output=True, text=True)
if esito.returncode != 0:
    raise RuntimeError("Non riesco a scaricare il progetto da GitHub. Se il repo è privato, controlla che il "
                       "secret GITHUB_TOKEN esista e che l'accesso dal notebook sia attivo.")

os.chdir(CARTELLA)
if CARTELLA not in sys.path:
    sys.path.insert(0, CARTELLA)
for nome in [m for m in sys.modules if m == "engine" or m.startswith("engine.")]:
    del sys.modules[nome]

%matplotlib inline
from IPython.utils.capture import capture_output
with capture_output():      # le librerie stampano avvisi tecnici al primo caricamento
    from engine.vetrina import (prepara, esplora, classifiche, scegli_trigger,
                                imposta_uscite, trova_strategia, scheda, stress_test)
print("Pronto.")

## 1 · Configura
Scegli lo strumento e per quante barre osservare il prezzo dopo ogni segnale (1 barra = 15 minuti).

In [ ]:
#@title 1 · Configura { display-mode: "form" }
strumento = "EURUSD"  #@param ["EURUSD", "BTCUSD"]
orizzonte = 25  #@param {type:"slider", min:4, max:96, step:1}

s = prepara(strumento, H=orizzonte)

## 2 · Esplora
Cosa fa il prezzo dopo ogni segnale di ingresso?

In [ ]:
#@title 2 · Esplora { display-mode: "form" }
esplora(s);

## 3 · Scegli i trigger
Le due classifiche, e poi la tua scelta.

In [ ]:
#@title 3a · Le classifiche { display-mode: "form" }
classifiche(s);

In [ ]:
#@title 3b · La tua scelta { display-mode: "form" }
#@markdown Copia i nomi dalla classifica. Più nomi sullo stesso lato: separali con una virgola. Lato vuoto = spento.
trigger_long = "E11_INVERTED_HAMMER, E1_RSI_CROSS_OVERSOLD"  #@param {type:"string"}
trigger_short = "E9_SHORT_CLOSING_PATTERN_ONLY_II, E4_SHORT_EMA_CROSS_DOWN"  #@param {type:"string"}

scegli_trigger(s, long=trigger_long, short=trigger_short)

## 4 · Imposta le uscite
Dopo quante barre chiudere, e quanto larghi tenere stop e target.

In [ ]:
#@title 4 · Imposta le uscite { display-mode: "form" }
#@markdown Uscita a tempo, in barre (può essere diversa fra long e short)
barre_long = 10  #@param {type:"slider", min:1, max:96, step:1}
barre_short = 18  #@param {type:"slider", min:1, max:96, step:1}
#@markdown Stop e target adattivi, come percentile (0 = spento)
stop = 90  #@param {type:"slider", min:0, max:99, step:1}
target = 0  #@param {type:"slider", min:0, max:99, step:1}

imposta_uscite(s, barre_long=barre_long, barre_short=barre_short, sl=stop, tp=target);

## 5 · Trova la strategia
Lo strumento prova tutti i filtri disponibili sul tuo sistema base. Richiede circa mezzo minuto.

In [ ]:
#@title 5 · Trova la strategia { display-mode: "form" }
trova_strategia(s);

## 6 · Scheda strategia
La prova sui dati che la strategia non ha mai visto.

In [ ]:
#@title 6 · Scheda strategia { display-mode: "form" }
#@markdown Nome del filtro copiato dalla classifica. Vuoto = sistema base, senza filtro.
filtro = ""  #@param {type:"string"}

scheda(s, filtro=filtro);

## 7 · Stress test
Quanto drawdown aspettarsi, e con quanti lotti partire.

In [ ]:
#@title 7 · Stress test { display-mode: "form" }
drawdown_massimo_in_dollari = 10000  #@param {type:"number"}

stress_test(s, drawdown_max_usd=drawdown_massimo_in_dollari);